### Imports

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


from data.utils import split_forget_retain, split_random
from data.dataloaders import unmark_dataset
import time
from unlearn.utils import do_unlearning
from trainer.utils import init_folder_if_not_exists

### Set configs for the experiment

In [ ]:

device = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"
exp_config = {

    "description": "First Test of Auditor Needs",
    
    "device": device,
    "model_class": "ResNet",
    "num_runs": 1,
    "retrain_from_scratch": True,
    "train_base": False,
    "measure_base_results": True,

    "data": {
        "dataset": "CIFAR10",
        "num_classes": 10,
        "batch_size": 512,
        "num_workers": 4,
        },

    "training": {
        "num_epochs": 30,
        "learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "batch_print_freq": 5,
        },
    
    "unlearning": {
        "methods": ["FT"],
        # "metrics": ["forget_acc", "retain_acc", "test_acc", "MIA"],
        "num_epochs": 10,
        "measure_every": 1,
        "save_checkpoints_at": [10],
        "classes_to_unlearn": [5],
        "percents_to_unlearn": None,
        "learning_rate": {
            "GA": 5e-5, # 5e-5# recall we are now doing SGD, so the learning rate is different from Adam
            "FT": 1e-4, # Maybe 1e-3 is just too high for SVHN
            "boundary_shrink": 1e-5, # from original paper
            "bad_teacher": 5e-5,
            "scrub": 1e-5 # 5e-4 is actually what they use for one "round"
            },
        "batch_print_freq": {
            "GA": 1,
            "FT": 8,
            "boundary_shrink": 1,
            "bad_teacher": 3,
            "scrub": 3,
            }
        }
}

### Protocol for several runs

In [3]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/jerrymoncus/.netrc.
wandb: Currently logged in as: jjmoncus (jjmoncus706) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
import glob
from models.archs.utils import init_model
from torch.optim.lr_scheduler import ReduceLROnPlateau
from trainer.utils import training_regimen_lr_annealing
from data.dataloaders import load_dataloaders_for_experiment
from evaluation.utils import measure_unlearning_metrics
import json

def run_experiment(config, results_folder, checkpoint_folder):
    
    print("="*70)
    print("="*19 + "  " + f'RUNNING EXPERIMENT, SEED {config["GRAND_SEED"]}' + "  " + "="*19)
    print("="*70 + "\n")

    # Make experiment results folder if it doesnt already exist
    if not os.path.exists(results_folder):
        print(f"{results_folder} doesn't exist - creating it...\n")
        os.makedirs(results_folder, exist_ok=True)

    # Save the config for this experiment to the main results folder
    with open(os.path.join(results_folder, "experiment_config.json"), "w") as f:
        json.dump(config, f, indent=4)

    # create a subfolder for saving model checkpoints for this experiment
    print(f'All models will be of class {config["model_class"]}.\n')
    checkpoint_subfolder = os.path.join(checkpoint_folder, f"seed_{config['GRAND_SEED']}")
    if not os.path.exists(checkpoint_subfolder):   
        print(f"{checkpoint_subfolder} doesn't exist - creating it...\n")
        os.makedirs(checkpoint_subfolder, exist_ok=True)

    # ----------------------------------------------------------------------------------- #
    # ----------------------------------------------------------------------------------- #
    # ---------------- TRAIN A BASE MODEL, FROM WHICH UNLEARNING BEGINS ----------------- #
    # ----------------------------------------------------------------------------------- #
    # ----------------------------------------------------------------------------------- #


    # log base model items to wandb (regardless of whether we're training or just evaluating metrics)
    wandb.init(
        project="Verifying-Unlearning-2026",
        name=f"{config['GRAND_SEED']}_base",
        config=config,
        reinit= "finish_previous"
    )

    # If you want to train your base model, ...
    if config["train_base"]: 


        print("-"*57)
        print("-"*13 + "  " + f"TRAINING NEW BASE MODEL" + "  " + "-"*13)
        print("-"*57 + "\n")

        # get some data
        full_train, _, full_test = load_dataloaders_for_experiment(
            name = config["data"]["dataset"],
            batch_size=config["data"]["batch_size"], 
            num_workers=config["data"]["num_workers"], 
            seed=config["GRAND_SEED"], 
            class_to_replace=None, 
            percent_to_replace=None,
            val = False
            )

        # init model, opt, criterion, and scheduler
        empty_model = init_model(model_class = config["model_class"], num_classes = config["num_classes"]).to(config["device"])
        opt = optim.Adam(empty_model.parameters(), lr=config["training"]["learning_rate"], weight_decay = config["training"]["weight_decay"])
        criterion = nn.CrossEntropyLoss()
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
                    opt, 
                    T_max=config["training"]["num_epochs"], 
                    eta_min=1e-6
                    )
        
        # train
        base_model_path = os.path.join(checkpoint_subfolder, "base_model.pth")
        base_model, opt, scheduler, train_loss, train_acc, train_entr, train_m_entr = training_regimen_lr_annealing(
            empty_model, 
            full_train,
            opt, 
            criterion, 
            scheduler, 
            device = config["device"], 
            num_epochs=config["training"]["num_epochs"], 
            model_path = base_model_path,
            print_freq = config["training"]["batch_print_freq"],
            w_and_b = True
            )
        
        print(f"base model successfully trained.\n")

    # Otherwise, pull a good base model from somewhere
    else:
        
        print("-"*57)
        print("-"*5 + "  " + f"NOT TRAINING BASE MODEL - PULLING INSTEAD" + "  " + "-"*5)
        print("-"*57 + "\n")

        all_paths = glob.glob(os.path.join("models/model_checkpoints/pretrained/seed_1/30_epochs", "*.pth"))
        base_model_path = [f for f in all_paths if config["model_class"] in f][0] # janky way of only grabbing the first element
        base_model = init_model(model_class = config["model_class"], num_classes = config["num_classes"], checkpoint_path = base_model_path).to(config["device"])
        print(f"base model successfully loaded from {base_model_path}.\n")
    
    # and init a subfolder for all results pertaining to the base model
    base_subfolder = init_folder_if_not_exists( os.path.join(results_folder, "base") )


    # ... decide if we're unlearning percents or classes (whichever one is non-empty)
    we_are_unlearning_classes = True if config["unlearning"]["classes_to_unlearn"] else False
    items_to_unlearn = config["unlearning"]["classes_to_unlearn"] if we_are_unlearning_classes else config["unlearning"]["percents_to_unlearn"]
    if not items_to_unlearn:
        raise ValueError("Either `classes_to_unlearn` or `percents_to_unlearn` need to be specified")
    
    # ----------------------------------------------------------------------------------- #
    # ----------------------------------------------------------------------------------- #
    # ----------------------------- DEFINE UNLEARNING LOADERS --------------------------- #
    # ----------------------------------------------------------------------------------- #
    # ----------------------------------------------------------------------------------- #
    

    # ... Loop through all the items we want to unlearn, 
    for c in items_to_unlearn:

        # ... announce what we're unlearning
        unlearn_name = f"class_{c}" if we_are_unlearning_classes else f"percent_{c}"
        print("-"*15 + "    " + "Forget set: " + unlearn_name + "\n")
        

        # ... be intelligent about setting `class_to_replace` or `percent_to_replace` if either is None
        class_param = c if we_are_unlearning_classes else None
        percent_param = c if not we_are_unlearning_classes else None
        

        # ...  ------------- get some unlearning data for this experiment ------------------- #
        # ... the dataSET is fixed across runs, and the randomness within runs is handled by simply shuffling the data loader. There is no need to actually apply the micro-seed

        # test is marked here, so we have to unmark them downstream
        marked_train_loader, _, marked_test_loader = load_dataloaders_for_experiment(
            name = config["data"]["dataset"],
            batch_size=config["data"]["batch_size"], 
            num_workers=config["data"]["num_workers"], 
            seed = config["GRAND_SEED"], 
            class_to_replace=class_param, 
            percent_to_replace=percent_param, 
            only_mark=True,
            val=False
            )
        # we make sure forget and retain sets are shuffled, to allow randomness across runs
        print("Training - forget vs retain split:")
        forget_loader, retain_loader = split_forget_retain(marked_train_loader, batch_size=config["data"]["batch_size"], shuffle = True, num_workers=config["data"]["num_workers"])
        
        
        # for datasets we're just evaling on, want shuffle = False
        print("Split 20 percent of `retain` for the MIAs...")
        retain_one_loader, retain_two_loader = split_random(retain_loader, p = .2, seed = config["GRAND_SEED"], batch_size=config["data"]["batch_size"], shuffle = False, num_workers=config["data"]["num_workers"])
        
        # unmark the test set
        unmark_dataset(marked_test_loader.dataset)
        
        unlearning_loaders = {
            "forget": forget_loader, # forget is always taken from train
            "retain": retain_loader,
            "test": marked_test_loader, # this is the FULL test set (now no longer marked)
            "retain_one": retain_one_loader, # This is passed as the TRAINING data to the MIA
            "retain_two": retain_two_loader # this is the TEST-TRAIN data for the MIA (to gut check that it indeed predicts "member" for these
        }

        if config["measure_base_results"]:
            # ... evaluate how good your base model is on this particular forget set
            print("Evaluating metrics on base model...\n")        

            base_name = f"base_{unlearn_name}"
            base_results, base_out = measure_unlearning_metrics(
                model = base_model, 
                dataloaders = unlearning_loaders, 
                device = config["device"]
                )
            base_results["type"] = "base"
            
            # ... save base results and pth out
            wandb.log(base_results)
            with open(os.path.join(base_subfolder, f"{base_name}.json"), "w") as f:
                json.dump(base_results, f, indent=4)

            torch.save(base_out, os.path.join(base_subfolder, f"{base_name}_out.pth"))

        # ...this closes the base model wandb session
        wandb.finish()

        # ----------------------------------------------------------------------------------- #
        # ----------------------------------------------------------------------------------- #
        # ------------------------------- DO SOME UNLEARNING -------------------------------- #
        # ----------------------------------------------------------------------------------- #
        # ----------------------------------------------------------------------------------- #

        print("-"*54)
        print("-"*15 + "  " + f"BEGINNING UNLEARNING" + "  " + "-"*15)
        print("-"*54 + "\n")
        
        # ... THEN, for each unlearning method, 
        for method in config["unlearning"]["methods"]:
        
            # ... and do a bunch of runs, where ...
            for i in range(1, config["num_runs"]+1):

                # ... open new wandb session per method (so that data for all runs is stored in one session)
                wandb.init(
                    project="Verifying-Unlearning-2026",
                    name=f"{config['GRAND_SEED']}_{method}_{unlearn_name}_run_{i}",
                    config=config,
                    reinit= "finish_previous"
                    )
                    
                print("="*25 + "    " + f"RUN {i}\n")

                # ----------------------------------------------------------------------------------- #
                # ----------------------------------------------------------------------------------- #
                # ----------------------- DO A BUNCH OF UNLEARNING METHODS -------------------------- #
                # ----------------------------------------------------------------------------------- #
                # ----------------------------------------------------------------------------------- #
                    
                # ... we need a new copy of the base model to begin unlearning each method on.
                # Instead of deepcopy:
                unlearn_model = init_model(model_class=config["model_class"], num_classes = config["num_classes"], checkpoint_path = None).to(config["device"]) # specify "None" in that it is empty, not pretrained
                unlearn_model.load_state_dict(base_model.state_dict()) # we do this to avoid the overhead of deepcopying the model before every run

                # ... has to be in eval mode I think (so BarchNorm layers aren't screwed)
                unlearn_model.eval()
                
                # ... actually doing the unlearning (results are written and saved out underneath this function)
                item_name = f"class_{c}" if we_are_unlearning_classes else f"percent_{c}"
                
                _ = do_unlearning(
                    base_results_folder = f"{results_folder}/unlearn/run_{i}",
                    
                    num_epochs = config["unlearning"]["num_epochs"],
                    unlearning_lr = config["unlearning"]["learning_rate"][method],
                    measure_every = config["unlearning"]["measure_every"],
                    device = config["device"],

                    method = method, # here, it is a string, and is converted to a function underneath
                    model = unlearn_model,
                    dataloaders = unlearning_loaders,
                    run = i,
                    forget_set_type = "class" if we_are_unlearning_classes else "percent",
                    unlearning_item = c,
                    w_and_b = True,
                    save_checkpoints_at = config["unlearning"]["save_checkpoints_at"],
                    checkpoint_subfolder = checkpoint_subfolder,
                    print_freq = config["unlearning"]["batch_print_freq"][method],

                    # we add a blank model, just in case we need it for bad_teacher or SCRUB
                    blank_model = init_model(model_class=config["model_class"], num_classes = config["num_classes"], checkpoint_path = None).to(config["device"]),
                    seed = config["GRAND_SEED"]
                    )
                
            # this closes the unlearning method wandb session
            wandb.finish()

        # ----------------------------------------------------------------------------------- #
        # ----------------------------------------------------------------------------------- #
        # ----------------------------- RETRAIN FROM SCRATCH -------------------------------- #
        # ----------------------------------------------------------------------------------- #
        # ----------------------------------------------------------------------------------- #

        # If you want to retrain from scatch, too ...
        if config["retrain_from_scratch"]:

            print(" -------------------- Starting retraining from scratch...\n")

            # ... open new wandb session per method (so that data for all runs is stored in one session)
            wandb.init(
                project="Verifying-Unlearning-2026",
                name=f"{config['GRAND_SEED']}_retrain_{unlearn_name}",
                config=config,
                reinit= "finish_previous"
                )
            # ... do a bunch of runs, where ...
            for i in range(1, config["num_runs"]+1):

                print(f" ----- Retraining from scratch for run {i}, {unlearn_name} ----- \n")
                
                # ... init a fresh model, opt, criterion, and scheduler
                empty_model = init_model(model_class = config["model_class"], num_classes = config["num_classes"]).to(config["device"])
                opt = optim.Adam(empty_model.parameters(), lr=config["training"]["learning_rate"], weight_decay = config["training"]["weight_decay"])
                criterion = nn.CrossEntropyLoss()
                scheduler = optim.lr_scheduler.CosineAnnealingLR(
                    opt, 
                    T_max=config["training"]["num_epochs"], 
                    eta_min=1e-6
                    )

                # ... do the training
                retrain_name = f"retrain_run_{i}_{unlearn_name}"
                retrain_checkpoint_path = os.path.join(checkpoint_subfolder, f"{retrain_name}.pth")
                start = time.time() # EVENTUALLY NEEDS TO BE MEASURED SOME OTHER WAY
                retrained_model, opt, scheduler, retrain_retain_loss, retrain_retain_acc, retrain_retain_entr, retrain_retain_m_entr = training_regimen_lr_annealing(
                    empty_model, 
                    retain_loader,
                    opt, 
                    criterion, 
                    scheduler, 
                    device = config["device"], 
                    num_epochs=config["training"]["num_epochs"], 
                    model_path = retrain_checkpoint_path,
                    print_freq = config["training"]["batch_print_freq"],
                    w_and_b = True
                    )
                end = time.time()
                wandb.log({"run time efficiency": end - start})
                
                # eval model on metrics
                retrained_results, retrained_out = measure_unlearning_metrics(
                    model = retrained_model, 
                    dataloaders = unlearning_loaders, 
                    device = config["device"],
                    )
                retrained_results.update({
                    "type": "retrain",
                    "run": i,
                    "forget_set_type": "class" if we_are_unlearning_classes else "percent",
                    "unlearning_item": c,
                    "method": "retrain"
                })

                # and init a subfolder for all results pertaining to the retrained models
                retrain_subfolder = init_folder_if_not_exists( os.path.join(results_folder, "retrain") )

                # save retrain results
                wandb.log(retrained_results)
                with open(os.path.join(retrain_subfolder, f"{retrain_name}.json"), "w") as f:
                    json.dump(retrained_results, f, indent=4)
                
                torch.save(retrained_out, os.path.join(retrain_subfolder, f"{retrain_name}_out.pth"))

            # closes retrain wandb session
            wandb.finish()



    print("-"*70)
    print("-"*19 + "  " + f'FINISHED EXPERIMENT, SEED {config["GRAND_SEED"]}' + "  " + "-"*19)
    print("-"*70 + "\n")
    

### Check metrics on unlearned models

In [5]:
# MAKE A RANDOM SEED
exp_config["GRAND_SEED"] = 3015

# DO EXP
run_experiment(
    config = exp_config, 
    results_folder = f"results/seed_{exp_config['GRAND_SEED']}", 
    checkpoint_folder="models/model_checkpoints"
    )

===================  RUNNING EXPERIMENT, SEED 3015  ===================

All models will be of class ResNet.



---------------------------------------------------------
-----  NOT TRAINING BASE MODEL - PULLING INSTEAD  -----
---------------------------------------------------------

The normalize layer is contained in the network
base model successfully loaded from models/model_checkpoints/pretrained/seed_1/30_epochs/ResNet_3.pth.

---------------    Forget set: class_5



/opt/miniconda3/envs/comp0081/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


========== DATALOADER INFO
Dataset: CIFAR-10
Train: 50000 images for training
Test: 10000 images for testing
Replaced class 5 in train
Training augmentation = randomcrop(32,4) + randomhorizontalflip + colorjitter + randomrotation + normalize
Validation/Test augmentation = normalize
num_workers = 4


Training - forget vs retain split:
Forget set: 5000 items
Retain set: 45000 items


Split 20 percent of `retain` for the MIAs...
Split one: 36000 items
Split two: 9000 items

Evaluating metrics on base model...

Evaluating forget set metrics...



/opt/miniconda3/envs/comp0081/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


...

 ----- Retraining from scratch for run 1, class_5 ----- 

The normalize layer is contained in the network
 ----- EPOCH 1 ----- 



/opt/miniconda3/envs/comp0081/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch: [1][4/88]	Loss 2.8308 (4.7734)	Accuracy 24.023 (16.094)	Entropy 3.1069 (5.3325)	M-Entropy 2.9549 (4.8077)	Time 11.27
Epoch: [1][9/88]	Loss 1.8472 (3.4144)	Accuracy 33.203 (22.852)	Entropy 1.7976 (3.7202)	M-Entropy 1.7676 (3.4118)	Time 3.85
Epoch: [1][14/88]	Loss 1.7007 (2.8513)	Accuracy 35.547 (26.732)	Entropy 1.7307 (3.0653)	M-Entropy 1.6005 (2.8153)	Time 3.40
Epoch: [1][19/88]	Loss 1.5646 (2.5383)	Accuracy 41.602 (30.000)	Entropy 1.6121 (2.7165)	M-Entropy 1.4695 (2.4842)	Time 3.36
Epoch: [1][24/88]	Loss 1.5268 (2.3427)	Accuracy 41.992 (31.875)	Entropy 1.5200 (2.4882)	M-Entropy 1.4508 (2.2819)	Time 3.36
Epoch: [1][29/88]	Loss 1.4636 (2.2108)	Accuracy 46.875 (33.555)	Entropy 1.5167 (2.3287)	M-Entropy 1.3725 (2.1473)	Time 3.35
Epoch: [1][34/88]	Loss 1.4638 (2.1096)	Accuracy 43.945 (35.039)	Entropy 1.4423 (2.2062)	M-Entropy 1.4068 (2.0461)	Time 3.35
Epoch: [1][39/88]	Loss 1.4431 (2.0303)	Accuracy 44.922 (36.118)	Entropy 1.5014 (2.1174)	M-Entropy 1.3559 (1.9659)	Time 3.35
Epoch: [1

RAM_GB,██▆▇▆█▆▇▇▇▇▇▇▇▇▇▇▆▆▇▇▆▁▄▇▄▇▃▇▇
VRAM_GB,▁█████████████████████████████
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
forget_acc,▁
forget_entr,▁
forget_loss,▁
forget_m_entr,▁
learning_rate,██████▇▇▇▇▆▆▆▅▅▄▄▄▃▃▃▂▂▂▂▁▁▁▁▁
retain_acc,▁
retain_entr,▁
+19,...


----------------------------------------------------------------------
-------------------  FINISHED EXPERIMENT, SEED 3015  -------------------
----------------------------------------------------------------------

